In [2]:
import warnings
warnings.filterwarnings("ignore")          
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline               
import torch

## 1. Model Loading

In [3]:
MODEL_NAME = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval() 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(DEVICE)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


## 2. Response Generation Function

In [4]:
def generate_response(user_input: str, chat_history_ids=None, max_new_tokens: int = 150) -> tuple:
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"         
    ).to(DEVICE)
    if chat_history_ids is not None:
        input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        input_ids = new_input_ids
    MAX_CONTEXT = 900
    if input_ids.shape[-1] > MAX_CONTEXT:
        input_ids = input_ids[:, -MAX_CONTEXT:]
    with torch.no_grad():          
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,       
            temperature=0.75,      
            top_p=0.92,            
            top_k=50,              
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,  
        )
    response_ids = output_ids[:, input_ids.shape[-1]:]
    response_text = tokenizer.decode(response_ids[0], skip_special_tokens=True).strip()
    return response_text, output_ids

## 3. Interactive Chatbot Loop

In [ ]:
def run_chatbot():
    print("=" * 60)
    print("  🤖  AI Chatbot  |  Powered by DialoGPT-medium")
    print("=" * 60)
    print("  Hello! I am your AI assistant. How can I help you today?")
    print("  (Type 'exit' or 'quit' to end the conversation)")
    print("-" * 60)
    chat_history_ids = None      
    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            # Graceful exit if the notebook kernel interrupts
            print("\nChatbot: Goodbye! Have a great day! 👋")
            break
        if not user_input:
            print("Chatbot: (Please type something!)")
            continue
        if user_input.lower() in {"exit", "quit"}:
            print("Chatbot: It was great talking to you. Goodbye! 👋")
            print("-" * 60)
            break
        response, chat_history_ids = generate_response(user_input, chat_history_ids)
        if not response:
            response = "I'm not sure how to respond to that. Could you rephrase?"
        print(f"Chatbot: {response}")
        print()    # blank line for readability
run_chatbot()

  🤖  AI Chatbot  |  Powered by DialoGPT-medium
  Hello! I am your AI assistant. How can I help you today?
  (Type 'exit' or 'quit' to end the conversation)
------------------------------------------------------------


You:  Hello


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: hi! how are you? :D what do u like to read on your day off?

